# Baseline models and rolling 28-day backtesting

Models receive `ForecastInputTask.history`; task objects do not contain target observations. Forecast outputs retain both raw and nonnegative-clipped values. Evaluation is performed by `track1/code/evaluate_predictions.py` after forecasts have been generated. Set the working directory to the repository root. This notebook demonstrates the baseline implementation included in this repository, not every later server experiment.

In [ ]:
from pathlib import Path
from datetime import date
import sys
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value): print(value.to_string() if hasattr(value, 'to_string') else value)
PROJECT = Path.cwd()
CODE = PROJECT / 'track1' / 'code'
assert CODE.is_dir(), 'Set the working directory to the repository root.'
sys.path.insert(0, str(CODE))
from data_pipeline import load_all, make_input_task
from models import MODEL_NAMES, forecast
from run_backtest import run_one, run
print('Available models:', ', '.join(MODEL_NAMES))


## Model provenance and scope

Seasonal Naive is the required baseline; ETS is an additional statistical baseline. SARIMA is inspired by the P1 `Fighting BI Team` approach. LightGBM is adapted from `6104_CJB`, and conventional regression models are adapted from `Machine Not Learning` / `Parrot Nothing`. The Transformer is a small, locally trained model written for numerical series; it is not a pretrained foundation model.

In [ ]:
families = pd.DataFrame([
    ('seasonal_naive','Statistical baseline','Repeat the last week'),
    ('ets','Additional statistical baseline','Additive 7-day seasonality'),
    ('sarima','Adapted from Fighting BI Team','Weekly seasonal statistical model'),
    ('lightgbm','Adapted from 6104_CJB','One model per horizon'),
    ('random_forest','Adapted from Machine Not Learning','One model per horizon'),
    ('parrot_random_forest_800','Adapted from Parrot Nothing','800 trees; one model per horizon'),
    ('ridge / linear_regression / gradient_boosting / svr / lasso / elasticnet','Conventional regression comparison','One model per horizon'),
    ('transformer','Written for numerical series','56-day input; 28-day output'),
], columns=['Model','Provenance','Input/output design'])
display(families)


## Single-task demonstration

The market and origin can be changed. This example uses the 180 days available at the origin to forecast the next 28 days; future observations are not passed to the model. The default demonstration uses Seasonal Naive, ETS and SARIMA. Add model names to `demo_models` to run LightGBM, Random Forest or the local Transformer, with their dependencies installed.

In [ ]:
series = load_all()
task = make_input_task(series['Macau:CN_Chinese_mainland'], date(2026, 6, 14), history_length=180)
demo_models = ['seasonal_naive', 'ets', 'sarima']
forecast_rows, time_rows = [], []
for name in demo_models:
    rows, timing = run_one(task, name)
    forecast_rows.extend(rows)
    time_rows.append(timing)
predictions = pd.DataFrame(forecast_rows)
display(predictions[['model','target_date','horizon_day','prediction_raw','prediction','was_clipped']].head(12))
display(pd.DataFrame(time_rows)[['model','fit_seconds','inference_seconds','total_seconds']])
assert predictions.groupby('model').size().eq(28).all()
assert 'actual' not in predictions.columns


## Inspect the four forecast horizons used in the paper

At this stage, inspect predictions only. MASE, RMSSE, absolute errors and high-demand-day metrics are calculated by the independent evaluator after joining the isolated targets.

In [ ]:
key_horizons = predictions[predictions['horizon_day'].isin([1,7,14,28])]
display(key_horizons[['model','horizon_day','target_date','prediction_raw','prediction','was_clipped']])


## Baseline rolling backtest (disabled by default)

Outer evaluation requires completion of B-group tuning and a frozen baseline configuration. When enabled, the cell below calls `baseline_config.validate(...)` to check the supplied configuration and task manifest, then requires the returned freeze status to be `frozen_after_B_group_tuning`. The notebook does not assume the current freeze status. The switch is disabled by default, and batch execution is rejected if validation fails or the required status is absent. This run covers only the selected baseline models below.

In [ ]:
RUN_PAPER_BACKTEST = False
if RUN_PAPER_BACKTEST:
    from baseline_config import validate
    audit = validate(task_manifest_path=PROJECT / 'shared' / 'data' / 'task_manifest.json')
    if audit['freeze_status'] != 'frozen_after_B_group_tuning':
        raise RuntimeError('Complete B-group tuning and freeze the baseline configuration before running outer evaluation.')
    selected = ('seasonal_naive','ets','sarima','random_forest','lightgbm','transformer')
    for destination in ('Macau','HongKong'):
        result = run(inputs=PROJECT / 'shared' / 'data' / 'task_inputs.parquet',
                     model_names=selected, destinations=(destination,),
                     history_lengths=('all',), splits=('outer_backtest',),
                     output=PROJECT / 'track1' / 'results' / destination.lower())
        print(destination, result['forecast_tasks'], result['prediction_rows'])


**Scope:** This notebook does not invoke TimesFM, Chronos, Moirai or multi-agent services, and it does not generate holiday/concert labels or cloud-instance billing records. The small Transformer used here is a locally trained model. Reproducing later server experiments requires their corresponding implementations, configurations, checkpoints and archived inputs/outputs; this notebook alone is not a reproduction entry point for all manuscript results.